In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings("ignore")

CLEANED_PATH = r"D:\Projects\End-to-end projects\17. SKU Proliferation & Rationalization\Data\Cleaned"
OUTPUT_PATH  = r"D:\Projects\End-to-end projects\17. SKU Proliferation & Rationalization\outputs"

sku_master = pd.read_csv(f"{CLEANED_PATH}/sku_master_clean.csv", parse_dates=["launch_date"])
sales      = pd.read_csv(f"{CLEANED_PATH}/sales_clean.csv",      parse_dates=["order_date"])
sku_ops    = pd.read_csv(f"{CLEANED_PATH}/sku_ops_clean.csv")
cfp        = pd.read_csv(f"{CLEANED_PATH}/customer_first_purchase_clean.csv")

sales["year_month"] = sales["order_date"].dt.to_period("M")
sales["revenue"]    = sales["units_sold"] * sales["selling_price"]

print("Data loaded ✓")

Data loaded ✓


In [2]:
# SKU-level revenue aggregation
sku_base = (
    sales
    .groupby("sku_id")
    .agg(
        total_revenue  = ("revenue",      "sum"),
        total_units    = ("units_sold",   "sum"),
        total_orders   = ("order_id",     "count"),
        avg_discount   = ("discount_pct", "mean"),
        avg_sell_price = ("selling_price","mean")
    )
    .reset_index()
)

# Merge master + ops
sku_base = (
    sku_base
    .merge(sku_master[["sku_id","category","subcategory",
                        "mrp","cost_price","gross_margin_pct",
                        "sku_age_months","is_active"]], on="sku_id")
    .merge(sku_ops[["sku_id","monthly_storage_cost","min_order_quantity",
                    "avg_handling_cost_per_unit","packaging_complexity_score",
                    "supplier_lead_time_days","monthly_complexity_cost"]], on="sku_id")
)

print(f"SKU base table: {sku_base.shape[0]} SKUs")
sku_base.head(3)

SKU base table: 258 SKUs


,sku_id,total_revenue,total_units,total_orders,avg_discount,avg_sell_price,category,subcategory,mrp,cost_price,gross_margin_pct,sku_age_months,is_active,monthly_storage_cost,min_order_quantity,avg_handling_cost_per_unit,packaging_complexity_score,supplier_lead_time_days,monthly_complexity_cost
0,SKU0001,425894.0,2076,1290,0.115426,205.221085,Haircare,Shampoo,232.0,78.0,66.38,16.0,True,510.0,100,9.79,2,14,910.0
1,SKU0002,1327884.4,1556,988,0.117105,853.759211,Haircare,Shampoo,967.0,277.0,71.35,7.3,True,336.0,100,44.44,4,14,1136.0
2,SKU0003,1671828.4,2043,1289,0.118231,819.163227,Haircare,Shampoo,929.0,393.0,57.70,37.6,True,335.0,100,14.19,1,21,535.0


In [3]:
# Sort by revenue descending, calculate cumulative %
sku_base = sku_base.sort_values("total_revenue", ascending=False).reset_index(drop=True)
sku_base["cumulative_rev_pct"] = (
    sku_base["total_revenue"].cumsum() / sku_base["total_revenue"].sum() * 100
)

# ABC rules:
# A = top SKUs driving first 70% of revenue
# B = next band up to 90%
# C = remaining 10%
def assign_abc(cum_pct):
    if cum_pct <= 70:
        return "A"
    elif cum_pct <= 90:
        return "B"
    else:
        return "C"

sku_base["abc_class"] = sku_base["cumulative_rev_pct"].apply(assign_abc)

abc_summary = (
    sku_base.groupby("abc_class")
    .agg(sku_count=("sku_id","count"),
         total_revenue=("total_revenue","sum"))
    .reset_index()
)
abc_summary["revenue_share"] = (
    abc_summary["total_revenue"] / abc_summary["total_revenue"].sum() * 100
).round(1)

print("=== ABC CLASSIFICATION ===\n")
print(abc_summary[["abc_class","sku_count","revenue_share"]].to_string(index=False))

# ABC score: A=3, B=2, C=1
abc_score_map = {"A": 3, "B": 2, "C": 1}
sku_base["score_abc"] = sku_base["abc_class"].map(abc_score_map)

=== ABC CLASSIFICATION ===

abc_class  sku_count  revenue_share
        A         38           69.3
        B         77           20.7
        C        143           10.0


In [4]:
# Monthly revenue per SKU
monthly_sku = (
    sales
    .groupby(["sku_id", "year_month"])["revenue"]
    .sum()
    .reset_index()
)

# Coefficient of Variation (CV) = std / mean
# Low CV = stable demand (X), High CV = erratic (Z)
cv_table = (
    monthly_sku
    .groupby("sku_id")["revenue"]
    .agg(["mean", "std"])
    .reset_index()
)
cv_table.columns = ["sku_id", "monthly_avg_rev", "monthly_std_rev"]
cv_table["cv"] = (cv_table["monthly_std_rev"] / cv_table["monthly_avg_rev"]).round(3)
cv_table["cv"] = cv_table["cv"].fillna(1.0)  # Single-month SKUs = high variance

# XYZ rules:
# X = CV < 0.5  → predictable
# Y = CV 0.5–1.0 → moderate variation
# Z = CV > 1.0  → erratic / unpredictable
def assign_xyz(cv):
    if cv < 0.5:
        return "X"
    elif cv <= 1.0:
        return "Y"
    else:
        return "Z"

cv_table["xyz_class"] = cv_table["cv"].apply(assign_xyz)

xyz_summary = cv_table["xyz_class"].value_counts().reset_index()
xyz_summary.columns = ["xyz_class","sku_count"]
print("=== XYZ CLASSIFICATION ===\n")
print(xyz_summary.to_string(index=False))

# Also add velocity tag from EDA
h1 = sales[sales["order_date"].between("2024-01-01","2024-06-30")]
h2 = sales[sales["order_date"].between("2025-07-01","2025-12-31")]
rev_h1 = h1.groupby("sku_id")["revenue"].sum().reset_index().rename(columns={"revenue":"rev_h1"})
rev_h2 = h2.groupby("sku_id")["revenue"].sum().reset_index().rename(columns={"revenue":"rev_h2"})
velocity = rev_h1.merge(rev_h2, on="sku_id", how="outer").fillna(0)
velocity["growth_pct"] = (
    (velocity["rev_h2"] - velocity["rev_h1"]) /
    velocity["rev_h1"].replace(0, np.nan) * 100
).round(1)

def classify_velocity(row):
    if row["rev_h1"] == 0 and row["rev_h2"] > 0: return "New/Emerging"
    elif row["rev_h2"] == 0:                       return "Dead"
    elif row["growth_pct"] >= 20:                  return "Growing"
    elif row["growth_pct"] <= -20:                 return "Declining"
    else:                                           return "Stable"

velocity["velocity_tag"] = velocity.apply(classify_velocity, axis=1)

# Merge XYZ + velocity into base
sku_base = sku_base.merge(cv_table[["sku_id","cv","xyz_class"]], on="sku_id", how="left")
sku_base = sku_base.merge(velocity[["sku_id","growth_pct","velocity_tag"]], on="sku_id", how="left")

# XYZ score: X=3, Y=2, Z=1
xyz_score_map = {"X": 3, "Y": 2, "Z": 1}
sku_base["score_xyz"] = sku_base["xyz_class"].map(xyz_score_map).fillna(1)

=== XYZ CLASSIFICATION ===

xyz_class  sku_count
        Y        145
        X        113


In [5]:
# True margin logic:
# Step 1 — Gross profit per SKU (revenue - COGS)
sku_base["gross_profit"] = (
    sku_base["total_units"] *
    (sku_base["avg_sell_price"] - sku_base["cost_price"])
).round(2)

# Step 2 — Annual complexity cost per SKU
# monthly_complexity_cost × 24 months
sku_base["total_complexity_cost"] = (sku_base["monthly_complexity_cost"] * 24).round(2)

# Step 3 — Handling cost for all units sold
sku_base["total_handling_cost"] = (
    sku_base["total_units"] * sku_base["avg_handling_cost_per_unit"]
).round(2)

# Step 4 — True profit
sku_base["true_profit"] = (
    sku_base["gross_profit"] -
    sku_base["total_complexity_cost"] -
    sku_base["total_handling_cost"]
).round(2)

# Step 5 — True margin %
sku_base["true_margin_pct"] = (
    sku_base["true_profit"] / sku_base["total_revenue"] * 100
).round(2)

# True margin score: bucket into 1–3
def score_true_margin(pct):
    if pct >= 30:   return 3   # Healthy
    elif pct >= 10: return 2   # Acceptable
    else:           return 1   # Poor or negative

sku_base["score_margin"] = sku_base["true_margin_pct"].apply(score_true_margin)

# Summary
print("=== TRUE MARGIN DISTRIBUTION ===\n")
margin_dist = pd.cut(
    sku_base["true_margin_pct"],
    bins=[-999, 0, 10, 30, 999],
    labels=["Negative", "0–10%", "10–30%", "30%+"]
).value_counts().reset_index()
margin_dist.columns = ["Margin Band", "SKU Count"]
print(margin_dist.to_string(index=False))

neg_margin = (sku_base["true_margin_pct"] < 0).sum()
print(f"\nSKUs with negative true margin: {neg_margin}")
print(f"Avg true margin across all SKUs: {sku_base['true_margin_pct'].mean():.1f}%")

=== TRUE MARGIN DISTRIBUTION ===

Margin Band  SKU Count
       30%+        157
   Negative         44
     10–30%         42
      0–10%         15

SKUs with negative true margin: 44
Avg true margin across all SKUs: 17.3%


In [6]:
# Gateway: SKU's role in acquiring new customers
gateway = (
    cfp
    .groupby("first_sku_id")
    .agg(
        times_as_gateway = ("customer_id",        "count"),
        avg_ltv_90d      = ("ltv_90d",            "mean"),
        repeat_rate      = ("is_repeat_customer",  "mean")
    )
    .reset_index()
    .rename(columns={"first_sku_id": "sku_id"})
)

gateway["avg_ltv_90d"] = gateway["avg_ltv_90d"].round(0)
gateway["repeat_rate"] = (gateway["repeat_rate"] * 100).round(1)

# Gateway Score = times_as_gateway × repeat_rate
# High count + high repeat = most valuable gateway
gateway["gateway_score_raw"] = (
    gateway["times_as_gateway"] * (gateway["repeat_rate"] / 100)
).round(1)

# Normalise to 1–3 scale
def score_gateway(raw, p33, p66):
    if raw >= p66:   return 3
    elif raw >= p33: return 2
    else:            return 1

p33 = gateway["gateway_score_raw"].quantile(0.33)
p66 = gateway["gateway_score_raw"].quantile(0.66)

gateway["score_gateway"] = gateway["gateway_score_raw"].apply(
    lambda x: score_gateway(x, p33, p66)
)

# SKUs with zero first purchases get gateway score = 1
sku_base = sku_base.merge(
    gateway[["sku_id","times_as_gateway","avg_ltv_90d",
             "repeat_rate","gateway_score_raw","score_gateway"]],
    on="sku_id", how="left"
)
sku_base["score_gateway"]      = sku_base["score_gateway"].fillna(1)
sku_base["times_as_gateway"]   = sku_base["times_as_gateway"].fillna(0)
sku_base["gateway_score_raw"]  = sku_base["gateway_score_raw"].fillna(0)

print("=== GATEWAY SCORE DISTRIBUTION ===\n")
print(sku_base["score_gateway"].value_counts().sort_index().reset_index().to_string(index=False))

=== GATEWAY SCORE DISTRIBUTION ===

 score_gateway  count
             1     83
             2     82
             3     93


In [7]:
# Weighted composite score
# Weights reflect business priority:
# Revenue (ABC) matters most → 35%
# Margin matters a lot       → 30%
# Velocity (XYZ) matters     → 20%
# Gateway role matters       → 15%

sku_base["rationalization_score"] = (
    sku_base["score_abc"]     * 0.35 +
    sku_base["score_margin"]  * 0.30 +
    sku_base["score_xyz"]     * 0.20 +
    sku_base["score_gateway"] * 0.15
).round(3)

# Recommendation logic
def assign_recommendation(row):
    score  = row["rationalization_score"]
    abc    = row["abc_class"]
    margin = row["true_margin_pct"]
    g_score= row["score_gateway"]
    vel    = row["velocity_tag"]

    # Override rules — business logic trumps score
    # Rule 1: High gateway value → always Protect regardless of score
    if g_score == 3 and row["times_as_gateway"] >= 300:
        return "Protect"

    # Rule 2: A-class SKU → never Kill
    if abc == "A":
        return "Accelerate"

    # Rule 3: Negative margin + declining + low gateway → Kill
    if margin < 0 and vel == "Declining" and g_score == 1:
        return "Kill"

    # Score-based rules
    if score >= 2.5:
        return "Accelerate"
    elif score >= 1.8:
        return "Watch"
    elif score >= 1.3:
        return "Watch"
    else:
        return "Kill"

sku_base["recommendation"] = sku_base.apply(assign_recommendation, axis=1)

# Summary
print("=== RATIONALIZATION RECOMMENDATIONS ===\n")
rec_summary = (
    sku_base.groupby("recommendation")
    .agg(
        sku_count      = ("sku_id",        "count"),
        total_revenue  = ("total_revenue", "sum"),
        avg_true_margin= ("true_margin_pct","mean")
    )
    .reset_index()
    .sort_values("sku_count", ascending=False)
)
rec_summary["revenue_share_pct"]  = (
    rec_summary["total_revenue"] / rec_summary["total_revenue"].sum() * 100
).round(1)
rec_summary["avg_true_margin"]    = rec_summary["avg_true_margin"].round(1)

print(rec_summary[["recommendation","sku_count",
                    "revenue_share_pct","avg_true_margin"]].to_string(index=False))

=== RATIONALIZATION RECOMMENDATIONS ===

recommendation  sku_count  revenue_share_pct  avg_true_margin
         Watch        137               14.7             19.1
       Protect         51               77.0             51.3
          Kill         39                1.4            -54.0
    Accelerate         31                6.9             43.1


In [9]:
# When a new SKU launched in a subcategory,
# did existing SKUs in that subcategory lose revenue?

# Monthly revenue by subcategory × SKU
monthly_sub = (
    sales
    .merge(sku_master[["sku_id","subcategory","launch_date"]], on="sku_id")
    .groupby(["subcategory","sku_id","year_month","launch_date"])["revenue"]
    .sum()
    .reset_index()
)
monthly_sub["year_month"] = monthly_sub["year_month"].dt.to_timestamp()
monthly_sub["launch_date"] = pd.to_datetime(monthly_sub["launch_date"])

# Find subcategories with 3+ SKUs (enough to detect cannibalization)
subcat_sku_counts = sku_master.groupby("subcategory")["sku_id"].count()
rich_subcats      = subcat_sku_counts[subcat_sku_counts >= 3].index.tolist()

cannibal_results = []

for subcat in rich_subcats:
    subcat_skus = sku_master[sku_master["subcategory"] == subcat].copy()
    subcat_skus = subcat_skus.sort_values("launch_date")

    if len(subcat_skus) < 2:
        continue

    # Find newest SKU in subcategory
    newest_sku       = subcat_skus.iloc[-1]
    existing_skus    = subcat_skus.iloc[:-1]["sku_id"].tolist()
    launch_threshold = newest_sku["launch_date"]

    # Revenue of existing SKUs before and after newest launch
    sub_sales = monthly_sub[monthly_sub["subcategory"] == subcat]

    before = sub_sales[
        (sub_sales["sku_id"].isin(existing_skus)) &
        (sub_sales["year_month"] < launch_threshold)
    ]["revenue"].mean()

    after = sub_sales[
        (sub_sales["sku_id"].isin(existing_skus)) &
        (sub_sales["year_month"] >= launch_threshold)
    ]["revenue"].mean()

    if before and before > 0:
        change_pct = ((after - before) / before * 100).round(1)
        cannibal_results.append({
            "subcategory"     : subcat,
            "newest_sku"      : newest_sku["sku_id"],
            "launch_date"     : launch_threshold.date(),
            "existing_sku_count": len(existing_skus),
            "avg_rev_before"  : round(before, 0),
            "avg_rev_after"   : round(after, 0),
            "revenue_change_pct": change_pct,
            "cannibal_flag"   : change_pct <= -10
        })

if cannibal_results:
    cannibal_df = pd.DataFrame(cannibal_results).sort_values("revenue_change_pct")
else:
    # No before/after splits found within sales window
    # Create empty frame with correct columns so downstream cells don't break
    cannibal_df = pd.DataFrame(columns=[
        "subcategory","newest_sku","launch_date","existing_sku_count",
        "avg_rev_before","avg_rev_after","revenue_change_pct","cannibal_flag"
    ])
    print("Note: No within-window SKU launches detected for cannibalization analysis.")
    print("Cannibalization module will show 0 — this is a data window limitation, not a logic error.")

print("=== CANNIBALIZATION ANALYSIS ===\n")
print(f"Subcategories analysed    : {len(cannibal_df)}")
print(f"Cannibalization detected  : {cannibal_df['cannibal_flag'].sum()} subcategories")
print(f"\nTop 10 most affected subcategories:")
print(cannibal_df.head(10)[["subcategory","newest_sku","launch_date",
                              "revenue_change_pct","cannibal_flag"]].to_string(index=False))

Note: No within-window SKU launches detected for cannibalization analysis.
Cannibalization module will show 0 — this is a data window limitation, not a logic error.
=== CANNIBALIZATION ANALYSIS ===

Subcategories analysed    : 0
Cannibalization detected  : 0 subcategories

Top 10 most affected subcategories:
Empty DataFrame
Columns: [subcategory, newest_sku, launch_date, revenue_change_pct, cannibal_flag]
Index: []


In [10]:
import os

# Master scored table
sku_base.to_csv(f"{OUTPUT_PATH}/sku_rationalization_master.csv", index=False)

# Recommendation summary
rec_summary.to_csv(f"{OUTPUT_PATH}/recommendation_summary.csv", index=False)

# Cannibalization results
cannibal_df.to_csv(f"{OUTPUT_PATH}/cannibalization_analysis.csv", index=False)

# Kill list — actionable output
kill_list = (
    sku_base[sku_base["recommendation"] == "Kill"]
    [["sku_id","category","subcategory","total_revenue",
      "true_margin_pct","velocity_tag","abc_class",
      "times_as_gateway","rationalization_score"]]
    .sort_values("rationalization_score")
)
kill_list.to_csv(f"{OUTPUT_PATH}/kill_list.csv", index=False)

print("=== FILES SAVED ===")
print(f"sku_rationalization_master : {sku_base.shape[0]} rows")
print(f"recommendation_summary     : {rec_summary.shape[0]} rows")
print(f"cannibalization_analysis   : {cannibal_df.shape[0]} rows")
print(f"kill_list                  : {kill_list.shape[0]} SKUs flagged for removal")

=== FILES SAVED ===
sku_rationalization_master : 258 rows
recommendation_summary     : 4 rows
cannibalization_analysis   : 0 rows
kill_list                  : 39 SKUs flagged for removal


In [11]:
print("=== CORE ANALYSIS COMPLETE ===\n")

print("[Scoring Dimensions]")
print(f"  ABC (Revenue)   — A: {(sku_base['abc_class']=='A').sum()} | B: {(sku_base['abc_class']=='B').sum()} | C: {(sku_base['abc_class']=='C').sum()} SKUs")
print(f"  XYZ (Stability) — X: {(sku_base['xyz_class']=='X').sum()} | Y: {(sku_base['xyz_class']=='Y').sum()} | Z: {(sku_base['xyz_class']=='Z').sum()} SKUs")
print(f"  Negative true margin SKUs : {(sku_base['true_margin_pct'] < 0).sum()}")
print(f"  Gateway SKUs (score=3)    : {(sku_base['score_gateway'] == 3).sum()}")

print("\n[Recommendations]")
for _, row in rec_summary.iterrows():
    print(f"  {row['recommendation']:<12}: {row['sku_count']:>3} SKUs | "
          f"{row['revenue_share_pct']:>5}% revenue | "
          f"Avg margin: {row['avg_true_margin']:>5}%")

print(f"\n[Cannibalization]")
print(f"  Subcategories with detected cannibalization: {cannibal_df['cannibal_flag'].sum()}")

total_kill_rev = kill_list["total_revenue"].sum()
print(f"\n[Kill List]")
print(f"  SKUs recommended for removal : {len(kill_list)}")
print(f"  Revenue at risk from kills   : ₹{total_kill_rev:,.0f}")
print(f"  (This revenue needs to be weighed against complexity cost savings)")

print("\nReady for Step 7: Insights & Findings ✓")

=== CORE ANALYSIS COMPLETE ===

[Scoring Dimensions]
  ABC (Revenue)   — A: 38 | B: 77 | C: 143 SKUs
  XYZ (Stability) — X: 113 | Y: 145 | Z: 0 SKUs
  Negative true margin SKUs : 44
  Gateway SKUs (score=3)    : 93

[Recommendations]
  Watch       : 137 SKUs |  14.7% revenue | Avg margin:  19.1%
  Protect     :  51 SKUs |  77.0% revenue | Avg margin:  51.3%
  Kill        :  39 SKUs |   1.4% revenue | Avg margin: -54.0%
  Accelerate  :  31 SKUs |   6.9% revenue | Avg margin:  43.1%

[Cannibalization]
  Subcategories with detected cannibalization: 0

[Kill List]
  SKUs recommended for removal : 39
  Revenue at risk from kills   : ₹1,273,188
  (This revenue needs to be weighed against complexity cost savings)

Ready for Step 7: Insights & Findings ✓
